In [1]:
import pandas as pd
import numpy as np
import optuna
import json
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Load data
train = pd.read_csv('playground-series-s6e9/train.csv')
test = pd.read_csv('playground-series-s6e9/test.csv')
sample_sub = pd.read_csv('playground-series-s6e9/sample_submission.csv')

# Preprocessing
target = 'Will_Buy_EV'
features = [c for c in train.columns if c not in ['id', target]]

cat_cols = train[features].select_dtypes(include=['object']).columns.tolist()

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])

train[target] = train[target].map({'Yes': 1, 'No': 0})

X = train[features]
y = train[target]
X_test = test[features]

num_trials = 50
n_splits = 5


In [2]:
from catboost import CatBoostClassifier, Pool

def objective(trial):
    params = {
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'auto_class_weights': 'Balanced',
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'depth': trial.suggest_int('depth', 3, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 1e-8, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'verbose': False,
        'task_type': 'CPU'
    }
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    auc_scores = []
    
    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        train_pool = Pool(X_tr, y_tr)
        val_pool = Pool(X_val, y_val)
        
        model = CatBoostClassifier(**params, iterations=1000, early_stopping_rounds=50)
        model.fit(train_pool, eval_set=val_pool, use_best_model=True, verbose=False)
        preds = model.predict_proba(val_pool)[:, 1]
        auc = roc_auc_score(y_val, preds)
        auc_scores.append(auc)
        
    return np.mean(auc_scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=num_trials)
print("Best trial:", study.best_trial.params)

best_params = study.best_trial.params
best_params['loss_function'] = 'Logloss'
best_params['eval_metric'] = 'AUC'
best_params['auto_class_weights'] = 'Balanced'
best_params['verbose'] = False

with open('catboost_best_params.json', 'w') as f:
    json.dump(best_params, f)


[I 2026-09-11 10:13:23,993] A new study created in memory with name: no-name-daa1843a-3b9d-4978-9d86-0cc7382d3a3a


[I 2026-09-11 10:19:51,864] Trial 0 finished with value: 0.9370625880878493 and parameters: {'learning_rate': 0.0019821302144564243, 'depth': 7, 'l2_leaf_reg': 1.1625487985310086, 'random_strength': 1.400425369438797e-07, 'bagging_temperature': 0.15843615588385007, 'border_count': 69}. Best is trial 0 with value: 0.9370625880878493.


[I 2026-09-11 10:34:00,367] Trial 1 finished with value: 0.9401470730544753 and parameters: {'learning_rate': 0.005951872828586698, 'depth': 8, 'l2_leaf_reg': 4.030874324221144, 'random_strength': 9.241194438013508e-08, 'bagging_temperature': 0.6521996594905399, 'border_count': 66}. Best is trial 1 with value: 0.9401470730544753.


[I 2026-09-11 10:50:41,727] Trial 2 finished with value: 0.9407270774745327 and parameters: {'learning_rate': 0.015113344817723088, 'depth': 10, 'l2_leaf_reg': 0.08807639989108791, 'random_strength': 5.281833594933759e-06, 'bagging_temperature': 0.1497566051552336, 'border_count': 156}. Best is trial 2 with value: 0.9407270774745327.


[I 2026-09-11 10:59:15,153] Trial 3 finished with value: 0.9381361641004096 and parameters: {'learning_rate': 0.0028844740562715175, 'depth': 6, 'l2_leaf_reg': 0.022345826285347675, 'random_strength': 3.1613366593434094e-07, 'bagging_temperature': 0.8244869302701234, 'border_count': 80}. Best is trial 2 with value: 0.9407270774745327.


...

[I 2026-09-11 22:28:36,416] Trial 46 finished with value: 0.941779620281068 and parameters: {'learning_rate': 0.07024852104235604, 'depth': 4, 'l2_leaf_reg': 8.255904337766284e-07, 'random_strength': 1.0693912361369216e-08, 'bagging_temperature': 0.02618557307463417, 'border_count': 215}. Best is trial 39 with value: 0.941818184719246.


[I 2026-09-11 22:51:48,945] Trial 47 finished with value: 0.9416927221524748 and parameters: {'learning_rate': 0.05992707584355379, 'depth': 4, 'l2_leaf_reg': 2.8601379691153346e-07, 'random_strength': 1.6371418710414548e-06, 'bagging_temperature': 0.027183675966430683, 'border_count': 203}. Best is trial 39 with value: 0.941818184719246.


[I 2026-09-11 23:12:26,665] Trial 48 finished with value: 0.9417602650798129 and parameters: {'learning_rate': 0.07255424688169751, 'depth': 3, 'l2_leaf_reg': 5.062615376824912e-07, 'random_strength': 2.955386842264963e-08, 'bagging_temperature': 0.10054372137278969, 'border_count': 243}. Best is trial 39 with value: 0.941818184719246.


[I 2026-09-11 23:32:53,300] Trial 49 finished with value: 0.9417495152366199 and parameters: {'learning_rate': 0.0649750347508778, 'depth': 4, 'l2_leaf_reg': 6.094984678029226e-07, 'random_strength': 9.012785487489027e-08, 'bagging_temperature': 0.47985049599626584, 'border_count': 212}. Best is trial 39 with value: 0.941818184719246.


Best trial: {'learning_rate': 0.07246421689500382, 'depth': 4, 'l2_leaf_reg': 1.8689472746239991e-07, 'random_strength': 1.1788386414059152e-07, 'bagging_temperature': 0.5485921956539299, 'border_count': 230}


In [3]:
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))
fold_preds_df = pd.DataFrame({'id': test['id']})

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    train_pool = Pool(X_tr, y_tr)
    val_pool = Pool(X_val, y_val)
    test_pool = Pool(X_test)
    
    model = CatBoostClassifier(**best_params, iterations=2000, early_stopping_rounds=50)
    model.fit(train_pool, eval_set=val_pool, use_best_model=True, verbose=False)
    
    val_preds = model.predict_proba(val_pool)[:, 1]
    oof_preds[val_idx] = val_preds
    fold_test_preds = model.predict_proba(test_pool)[:, 1]
    test_preds += fold_test_preds / n_splits
    fold_preds_df[f'fold_{fold}'] = fold_test_preds

print("OOF AUC:", roc_auc_score(y, oof_preds))

pd.DataFrame({'id': train['id'], 'Will_Buy_EV': oof_preds}).to_csv('catboost_oof_predictions.csv', index=False)
fold_preds_df.to_csv('catboost_fold_predictions.csv', index=False)

sample_sub['Will_Buy_EV'] = test_preds
sample_sub.to_csv('submission_catboost.csv', index=False)


OOF AUC: 0.9418289921042649
